[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/26.%20The%20Gate%20Automation%20%26%20Damage%20Detection%20Problem/P26-Tier-7_executed.ipynb)

---


# 26. The Gate Automation and Damage Detection Problem

## Tier 7: Human–AI Symbiotic Partnership (the Centaur Model)

### Goal

This notebook represents a human–AI partnership for gate operations and damage detection. It models how an AI system can provide a recommendation and an explanation while a human operator makes a decision. It also demonstrates simple rules for cognitive-load estimation and trust adjustment.

The code is an illustrative simulation, not a trained AI system, a validated operational model, or a field study. Its numeric rules and outcomes are assumptions for teaching the interaction logic; they are not measured performance claims.

### Inputs and outputs

The inputs are facility and operator descriptions, scenario data, and a scenario type. The code produces a recommendation, an explanation, a simulated human decision, a simulated outcome, and updated trust records. A small set of scenarios is then summarized and visualized.

### Scope and modeling assumptions

- The operator's expertise, initial trust, and cognitive load are specified as data. The simulation does not infer them from observations.
- AI recommendations are generated by scenario-specific rules. The feature-importance and attention values are illustrative explanation fields; they are not computed by SHAP, an attention-based model, or a trained predictor.
- A random draw represents whether the operator follows the recommendation. No random seed is specified, so simulated decisions can vary between runs.
- Trust adjustment is a simple rule based on recent simulated performance and feedback. It does not train or update an AI model.
- The high-security example includes a convoy arrival as scenario data, but the decision rule does not reason about arrival time or construct a combined plan involving a backup gate, speed reduction, and scheduled maintenance. The simulation therefore cannot establish that such a plan is feasible or that it prevents a real failure.
- There are no human-only or AI-only baseline simulations in this notebook. The results cannot establish that collaboration outperforms either baseline.

### High-security scenario specification

The illustrative scenario describes a predicted 73% probability of Gate 3 motor failure within 8 hours, a VIP convoy arriving in 6 hours, and a proposed option to reduce gate speed by 30%. That option is assigned a 23% failure probability and a 40% processing-time increase. These values are inputs to an illustrative scenario, not outputs of a model run or empirical estimates.

### Method overview

The solution proceeds in the following order: define the facility and its operators, generate an explainable recommendation, simulate the human–AI interaction, summarize the scenarios, and visualize the summary. This preserves the notebook's human–AI collaboration approach; it does not solve a gate-scheduling optimization problem.


## 1. A simple mathematical description of the simulation

### Intuition

The simulation has three linked parts. First, a rule estimates the operator's cognitive load. Second, a probability rule determines whether the operator follows the AI recommendation. Third, a feedback rule adjusts trust after the simulated outcome. Writing these rules mathematically makes their assumptions visible and connects the code to the reported results.

### Cognitive-load rule

For operator \(i\), let \(b_i\) be the experience-based load value, \(c\) the scenario complexity, and \(n_i\) the number of active decisions in the operator's history. The simulated cognitive load is

$$
L_i = \min\left(1,\; b_i c + 0.1 n_i\right).
$$

The code uses \(b_i=0.8\) for a novice, \(0.5\) for an intermediate operator, and \(0.3\) for an expert. The result is capped at 1, the upper end of the load scale. In the implementation, `assess_cognitive_load` calculates \(L_i\) and stores it in `operator.cognitive_load`.

### Recommendation-following rule

Let \(T_i\) be the operator's trust in AI before a decision. The simulation adds rule-based adjustments for cognitive load, expertise relevance, and recommendation confidence or uncertainty, then clips the result to the interval \([0,1]\):

$$
p_i = \operatorname{clip}\left(T_i + a_L + a_E + a_C,\;0,\;1\right).
$$

Here, \(a_L\), \(a_E\), and \(a_C\) are the load, expertise, and confidence/uncertainty adjustments implemented in `_simulate_human_decision`. A draw from the uniform distribution on \([0,1)\) determines whether the recommendation is followed: it is followed when the draw is less than \(p_i\). Thus \(p_i\) is a simulation probability, not an observed reliance rate or a calibrated psychological model.

### Trust-update rule

After an interaction, simulated AI performance is assigned a value \(q_t\): \(1.0\) for a successful outcome and \(0.5\) for any other outcome. Let \(\bar q_t\) be the mean of the latest five performance values, or of all available values if fewer than five exist. The performance-based adjustment is

$$
d_t =
\begin{cases}
0.05, & \bar q_t > 0.8,\\
0, & 0.6 < \bar q_t \leq 0.8,\\
-0.1, & \bar q_t \leq 0.6.
\end{cases}
$$

If \(s_t\) is the feedback sentiment, the updated trust is

$$
T_{t+1} = \operatorname{clip}\left(T_t + d_t + 0.1s_t,\;0,\;1\right).
$$

The code implements this rule in `calibrate_trust`, using `numpy.mean` and `numpy.clip`. This is a transparent teaching rule, not a validated trust-calibration model. Because performance is recorded after each interaction, changing the order or outcome labels would change subsequent trust values.

### Computational cost

For \(N\) simulated interactions, the scenario loop performs a constant amount of work per interaction apart from history operations. The displayed implementation also filters an operator's decision history to count active tasks; that count can take time proportional to the length of that history. The five-value performance average is bounded by a window of at most five values. These are small illustrative simulations, not a large-scale performance benchmark.


In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Dict, List, Optional
from collections import defaultdict
import random

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")

## 2. Representing the facility and collaboration components

### Purpose

A data class groups related information into a named object. This makes the simulation easier to inspect than a collection of unrelated values: an operator has expertise and trust, a recommendation has an action and supporting information, and a protocol describes roles for a scenario.

The classes below define the data passed between the facility setup, recommendation, interaction, and analysis steps. They are data structures; they do not themselves make decisions or learn from experience.

In [ ]:
@dataclass
class HumanOperator:
    """A human operator participating in the simulated partnership."""

    operator_id: str
    experience_level: str
    cognitive_load: float
    trust_in_ai: float
    expertise_areas: List[str]
    decision_history: List[Dict[str, Any]] = field(default_factory=list)
    feedback_history: List[Dict[str, Any]] = field(default_factory=list)


@dataclass
class AIRecommendation:
    """A simulated AI recommendation and its associated information."""

    recommendation_id: str
    recommendation_type: str
    primary_action: str
    confidence: float
    uncertainty: float
    reasoning: List[str]
    supporting_data: Dict[str, Any]
    alternative_options: List[Dict[str, Any]]
    expected_outcomes: Dict[str, float]


@dataclass
class Explanation:
    """Illustrative explanation fields associated with a recommendation."""

    explanation_id: str
    explanation_type: str
    feature_importance: Dict[str, float]
    attention_weights: Dict[str, float]
    counterfactual_scenarios: List[Dict[str, Any]]
    natural_language_summary: str


@dataclass
class TrustCalibration:
    """Trust state and feedback history for one operator."""

    calibration_id: str
    current_trust_level: float
    ai_performance_history: List[float]
    human_feedback_history: List[Dict[str, Any]]
    uncertainty_communication: str
    confidence_adjustment: float


@dataclass
class CollaborationProtocol:
    """Roles and escalation information for a scenario type."""

    protocol_id: str
    scenario_type: str
    human_role: str
    ai_role: str
    decision_authority: str
    communication_style: str
    escalation_triggers: List[str]


@dataclass
class SymbioticInstance:
    """Facility-level data for the simulated human–AI partnership."""

    facility_id: str
    operators: List[HumanOperator]
    ai_systems: List[Dict[str, Any]]
    collaboration_protocols: List[CollaborationProtocol]
    current_scenario: str
    partnership_performance: Dict[str, float]
    learning_metrics: Dict[str, float]

## 3. Creating the illustrative facility

### Purpose and interpretation

This function creates the operators, AI-system descriptions, and scenario protocols used by the later cells. The numerical performance and learning fields are supplied example values; this notebook does not estimate them. The printed counts and expertise areas describe the constructed objects, not measured facility performance.

In [ ]:
def create_symbiotic_facility() -> SymbioticInstance:
    """Create the example facility, operators, systems, and protocols."""

    facility_id = "HIGH_SECURITY_FACILITY_001"

    operators = [
        HumanOperator(
            operator_id="OP001",
            experience_level="expert",
            cognitive_load=0.3,
            trust_in_ai=0.8,
            expertise_areas=[
                "security_coordination",
                "emergency_response",
                "vip_handling",
            ],
        ),
        HumanOperator(
            operator_id="OP002",
            experience_level="intermediate",
            cognitive_load=0.5,
            trust_in_ai=0.6,
            expertise_areas=[
                "maintenance_scheduling",
                "system_optimization",
            ],
        ),
        HumanOperator(
            operator_id="OP003",
            experience_level="novice",
            cognitive_load=0.7,
            trust_in_ai=0.4,
            expertise_areas=["monitoring", "data_analysis"],
        ),
    ]

    ai_systems = [
        {
            "system_id": "PREDICTIVE_AI_001",
            "capabilities": [
                "failure_prediction",
                "pattern_recognition",
                "trend_analysis",
            ],
            "accuracy": 0.89,
            "explainability": "high",
            "specialization": "mechanical_systems",
        },
        {
            "system_id": "OPTIMIZATION_AI_002",
            "capabilities": [
                "resource_allocation",
                "schedule_optimization",
                "scenario_analysis",
            ],
            "accuracy": 0.82,
            "explainability": "medium",
            "specialization": "operational_planning",
        },
        {
            "system_id": "SECURITY_AI_003",
            "capabilities": [
                "threat_detection",
                "risk_assessment",
                "coordination",
            ],
            "accuracy": 0.91,
            "explainability": "high",
            "specialization": "security_management",
        },
    ]

    protocols = [
        CollaborationProtocol(
            protocol_id="ROUTINE_001",
            scenario_type="routine",
            human_role="monitor",
            ai_role="advisor",
            decision_authority="collaborative",
            communication_style="advisory",
            escalation_triggers=[
                "high_uncertainty",
                "low_confidence",
                "safety_critical",
            ],
        ),
        CollaborationProtocol(
            protocol_id="EMERGENCY_002",
            scenario_type="emergency",
            human_role="decider",
            ai_role="information_provider",
            decision_authority="human",
            communication_style="directive",
            escalation_triggers=[
                "life_safety",
                "security_breach",
                "critical_failure",
            ],
        ),
        CollaborationProtocol(
            protocol_id="MAINTENANCE_003",
            scenario_type="maintenance",
            human_role="validator",
            ai_role="planner",
            decision_authority="collaborative",
            communication_style="collaborative",
            escalation_triggers=[
                "cost_threshold",
                "operational_impact",
                "safety_risk",
            ],
        ),
        CollaborationProtocol(
            protocol_id="SECURITY_004",
            scenario_type="security",
            human_role="coordinator",
            ai_role="analyst",
            decision_authority="human",
            communication_style="collaborative",
            escalation_triggers=[
                "threat_level_high",
                "multiple_incidents",
                "vip_involved",
            ],
        ),
    ]

    return SymbioticInstance(
        facility_id=facility_id,
        operators=operators,
        ai_systems=ai_systems,
        collaboration_protocols=protocols,
        current_scenario="routine",
        partnership_performance={
            "decision_quality": 0.85,
            "response_time": 0.78,
            "trust_calibration": 0.82,
            "learning_rate": 0.15,
        },
        learning_metrics={
            "human_feedback_integration": 0.73,
            "ai_improvement_rate": 0.18,
            "collaboration_efficiency": 0.81,
        },
    )


symbiotic_facility = create_symbiotic_facility()

print("Symbiotic facility created:")
print(f"- Facility: {symbiotic_facility.facility_id}")
print(f"- Human operators: {len(symbiotic_facility.operators)}")
print(f"- AI systems: {len(symbiotic_facility.ai_systems)}")
print(
    "- Collaboration protocols: "
    f"{len(symbiotic_facility.collaboration_protocols)}"
)

print("\nOperator expertise:")
for operator in symbiotic_facility.operators:
    areas = ", ".join(operator.expertise_areas)
    print(
        f"  {operator.operator_id}: "
        f"{operator.experience_level} - {areas}"
    )

## 4. Generating recommendations and explanations

### Purpose

`ExplainableAI` selects a system description and applies explicit rules to create a recommendation. For the Gate 3 scenario, the recommendation carries the failure-risk information and the alternative speed-reduction option. The explanation object presents illustrative feature weights, attention weights, and alternative scenarios so the information structure is visible to a learner.

These fields should not be mistaken for a faithful explanation of a trained model. The weights are fixed example values, and the recommendation itself is rule-generated. The code only describes the convoy arrival; it does not use that arrival time to choose or evaluate an operational plan.

In [ ]:
class ExplainableAI:
    """Rule-based recommendation generator with illustrative explanations."""

    def __init__(self, ai_systems: List[Dict[str, Any]]):
        self.ai_systems = ai_systems
        self.explanation_history: List[Explanation] = []

    def generate_recommendation(
        self,
        scenario_data: Dict[str, Any],
        scenario_type: str,
    ) -> AIRecommendation:
        """Create a scenario-specific recommendation using fixed rules."""

        if scenario_type == "security":
            specialization = "security_management"
        elif scenario_type == "maintenance":
            specialization = "mechanical_systems"
        else:
            specialization = None

        ai_system = next(
            (
                system
                for system in self.ai_systems
                if system["specialization"] == specialization
            ),
            None,
        )
        if ai_system is None:
            ai_system = self.ai_systems[0]

        if scenario_type == "security" and "gate_3_vibration" in scenario_data:
            vibration_data = scenario_data["gate_3_vibration"]

            recommendation = AIRecommendation(
                recommendation_id="REC_001",
                recommendation_type="maintenance",
                primary_action=(
                    "Schedule immediate maintenance for Gate 3 motor"
                ),
                confidence=0.73,
                uncertainty=0.27,
                reasoning=[
                    "Motor temperature increased 15% above baseline",
                    "Vibration frequency shows a characteristic bearing-wear signature",
                    "Similar patterns preceded 3 previous failures in this system",
                    "The scenario specifies a 73% failure probability within 8 hours",
                ],
                supporting_data={
                    "temperature_increase": 15.0,
                    "vibration_amplitude": vibration_data,
                    "historical_failures": 3,
                    "predicted_rul_hours": 8.0,
                    "predicted_failure_probability": 0.73,
                    "vip_convoy_arrival_hours": scenario_data.get(
                        "vip_convoy_arrival"
                    ),
                },
                alternative_options=[
                    {
                        "action": "Reduce gate speed by 30%",
                        "failure_probability": 0.23,
                        "processing_time_impact": 40.0,
                        "confidence": 0.68,
                    },
                    {
                        "action": (
                            "Activate backup gate and continue normal speed"
                        ),
                        "failure_probability": 0.73,
                        "processing_time_impact": 0.0,
                        "confidence": 0.45,
                    },
                ],
                expected_outcomes={
                    "predicted_failure_probability": 0.73,
                    "predicted_rul_hours": 8.0,
                    "cost_impact": 25000.0,
                },
            )

        elif scenario_type == "optimization":
            recommendation = AIRecommendation(
                recommendation_id="REC_002",
                recommendation_type="optimization",
                primary_action=(
                    "Optimize gate scheduling based on demand forecast"
                ),
                confidence=0.82,
                uncertainty=0.18,
                reasoning=[
                    "Demand forecast shows a 25% increase in the next 4 hours",
                    "Current gate utilization is 65%",
                    "The illustrative model predicts a 15% throughput improvement",
                    "Adjusted timing is associated with an illustrative 8% energy reduction",
                ],
                supporting_data={
                    "demand_increase": 25.0,
                    "current_utilization": 0.65,
                    "predicted_improvement": 15.0,
                    "energy_savings": 8.0,
                },
                alternative_options=[
                    {
                        "action": "Maintain current schedule",
                        "throughput_impact": 0.0,
                        "energy_impact": 0.0,
                        "confidence": 0.95,
                    }
                ],
                expected_outcomes={
                    "throughput_improvement": 15.0,
                    "energy_reduction": 8.0,
                    "cost_savings": 5000.0,
                },
            )

        else:
            recommendation = AIRecommendation(
                recommendation_id="REC_003",
                recommendation_type="monitoring",
                primary_action="Continue normal monitoring operations",
                confidence=0.90,
                uncertainty=0.10,
                reasoning=[
                    "The example assumes that systems are operating within normal parameters",
                    "The example assumes that no anomalies were detected in recent data",
                    "The example assumes stable conditions",
                ],
                supporting_data={
                    "system_health": 0.95,
                    "anomaly_count": 0,
                    "stability_score": 0.92,
                },
                alternative_options=[],
                expected_outcomes={
                    "continued_availability": 0.98,
                    "maintenance_requirement": 0.05,
                },
            )

        return recommendation

    def generate_explanation(
        self,
        recommendation: AIRecommendation,
    ) -> Explanation:
        """Create illustrative explanation fields for a recommendation."""

        if recommendation.recommendation_type == "maintenance":
            feature_importance = {
                "motor_temperature": 0.35,
                "vibration_amplitude": 0.28,
                "historical_failure_rate": 0.22,
                "operational_hours": 0.15,
            }
        elif recommendation.recommendation_type == "optimization":
            feature_importance = {
                "demand_forecast": 0.40,
                "current_utilization": 0.25,
                "energy_consumption": 0.20,
                "maintenance_schedule": 0.15,
            }
        else:
            feature_importance = {}

        attention_weights = {
            "sensor_data": 0.45,
            "historical_patterns": 0.30,
            "operational_context": 0.15,
            "external_factors": 0.10,
        }

        counterfactual_scenarios = []
        for option in recommendation.alternative_options:
            scenario = {
                "scenario": f"What if we: {option['action']}?",
                "expected_outcome": option,
            }
            if "failure_probability" in option:
                current_probability = recommendation.supporting_data.get(
                    "predicted_failure_probability"
                )
                if current_probability is not None:
                    scenario["reasoning"] = (
                        "The illustrative failure probability changes from "
                        f"{current_probability:.2f} to "
                        f"{option['failure_probability']:.2f}."
                    )
            counterfactual_scenarios.append(scenario)

        explanation = Explanation(
            explanation_id=f"EXP_{recommendation.recommendation_id}",
            explanation_type="illustrative",
            feature_importance=feature_importance,
            attention_weights=attention_weights,
            counterfactual_scenarios=counterfactual_scenarios,
            natural_language_summary="\n".join(recommendation.reasoning),
        )
        self.explanation_history.append(explanation)
        return explanation

## 5. Simulating human–AI interaction

### How the interaction works

`HumanAICollaboration` connects the facility, recommendation generator, protocol selection, cognitive-load rule, decision rule, and trust-update rule. The interaction record stores both the recommendation and the human decision so that later analysis can distinguish what the AI proposed from what the simulated operator chose.

The decision is stochastic: trust and rule-based adjustments set a probability, and `random.random()` samples whether the operator follows the AI. If the operator does not follow it, the simulation may select one of the listed alternatives or use a generic independent action. The outcome rules are illustrative and should not be interpreted as predictions of real gate performance.

In [ ]:
class HumanAICollaboration:
    """Simulate interactions under the notebook's Centaur-model rules."""

    def __init__(self, symbiotic_facility: SymbioticInstance):
        self.facility = symbiotic_facility
        self.explainable_ai = ExplainableAI(symbiotic_facility.ai_systems)
        self.collaboration_history: List[Dict[str, Any]] = []
        self.trust_calibrators: Dict[str, TrustCalibration] = {}

        for operator in self.facility.operators:
            self.trust_calibrators[operator.operator_id] = TrustCalibration(
                calibration_id=f"TC_{operator.operator_id}",
                current_trust_level=operator.trust_in_ai,
                ai_performance_history=[],
                human_feedback_history=[],
                uncertainty_communication="quantitative",
                confidence_adjustment=0.1,
            )

    def get_collaboration_protocol(
        self,
        scenario_type: str,
    ) -> CollaborationProtocol:
        """Return the matching protocol, using the routine protocol as fallback."""

        protocol = next(
            (
                item
                for item in self.facility.collaboration_protocols
                if item.scenario_type == scenario_type
            ),
            None,
        )
        if protocol is not None:
            return protocol

        routine_protocol = next(
            (
                item
                for item in self.facility.collaboration_protocols
                if item.scenario_type == "routine"
            ),
            None,
        )
        if routine_protocol is None:
            raise ValueError("A routine collaboration protocol is required.")
        return routine_protocol

    def assess_cognitive_load(
        self,
        operator: HumanOperator,
        scenario_complexity: float,
    ) -> float:
        """Calculate cognitive load from experience, complexity, and active tasks."""

        experience_loads = {
            "novice": 0.8,
            "intermediate": 0.5,
            "expert": 0.3,
        }
        if operator.experience_level not in experience_loads:
            raise ValueError(
                f"Unknown experience level: {operator.experience_level}"
            )

        experience_load = experience_loads[operator.experience_level]
        concurrent_tasks = sum(
            decision.get("status") == "active"
            for decision in operator.decision_history
        )
        adjusted_load = (
            experience_load * scenario_complexity
            + concurrent_tasks * 0.1
        )
        operator.cognitive_load = float(np.clip(adjusted_load, 0.0, 1.0))
        return operator.cognitive_load

    def calibrate_trust(
        self,
        operator: HumanOperator,
        ai_performance: float,
        human_feedback: Dict[str, Any],
    ) -> float:
        """Update simulated trust from recent performance and human feedback."""

        calibrator = self.trust_calibrators[operator.operator_id]
        calibrator.ai_performance_history.append(ai_performance)
        calibrator.human_feedback_history.append(human_feedback)
        operator.feedback_history.append(human_feedback)

        recent_performance = np.mean(
            calibrator.ai_performance_history[-5:]
        )
        if recent_performance > 0.8:
            trust_adjustment = 0.05
        elif recent_performance > 0.6:
            trust_adjustment = 0.0
        else:
            trust_adjustment = -0.1

        feedback_sentiment = human_feedback.get("sentiment", 0.0)
        trust_adjustment += feedback_sentiment * 0.1

        calibrator.current_trust_level = float(
            np.clip(
                calibrator.current_trust_level + trust_adjustment,
                0.0,
                1.0,
            )
        )
        operator.trust_in_ai = calibrator.current_trust_level
        return calibrator.current_trust_level

    def human_ai_interaction(
        self,
        operator: HumanOperator,
        scenario_data: Dict[str, Any],
        scenario_type: str,
    ) -> Dict[str, Any]:
        """Generate one recommendation, decision, outcome, and trust update."""

        protocol = self.get_collaboration_protocol(scenario_type)
        scenario_complexity = scenario_data.get("complexity", 0.5)
        cognitive_load = self.assess_cognitive_load(
            operator,
            scenario_complexity,
        )

        ai_recommendation = self.explainable_ai.generate_recommendation(
            scenario_data,
            scenario_type,
        )
        explanation = self.explainable_ai.generate_explanation(
            ai_recommendation
        )
        trust_before = operator.trust_in_ai

        human_decision = self._simulate_human_decision(
            operator=operator,
            ai_recommendation=ai_recommendation,
            explanation=explanation,
            protocol=protocol,
            cognitive_load=cognitive_load,
            scenario_type=scenario_type,
        )

        interaction_record = {
            "timestamp": datetime.now(),
            "operator_id": operator.operator_id,
            "scenario_type": scenario_type,
            "protocol": protocol.protocol_id,
            "ai_recommendation": ai_recommendation,
            "explanation": explanation,
            "human_decision": human_decision,
            "cognitive_load": cognitive_load,
            "trust_before": trust_before,
        }

        interaction_outcome = human_decision["outcome"]
        ai_performance = (
            1.0 if interaction_outcome == "successful" else 0.5
        )
        human_feedback = {
            "decision": human_decision["action"],
            "agreement_with_ai": human_decision["agreement_with_ai"],
            "sentiment": human_decision["sentiment"],
            "confidence": human_decision["confidence"],
        }
        new_trust = self.calibrate_trust(
            operator,
            ai_performance,
            human_feedback,
        )

        interaction_record["trust_after"] = new_trust
        interaction_record["trust_change"] = new_trust - trust_before
        self.collaboration_history.append(interaction_record)
        operator.decision_history.append(interaction_record)
        return interaction_record

    def _simulate_human_decision(
        self,
        operator: HumanOperator,
        ai_recommendation: AIRecommendation,
        explanation: Explanation,
        protocol: CollaborationProtocol,
        cognitive_load: float,
        scenario_type: str,
    ) -> Dict[str, Any]:
        """Sample a decision and assign an illustrative scenario outcome."""

        ai_follow_probability = operator.trust_in_ai

        if cognitive_load > 0.7:
            ai_follow_probability += 0.2
        elif cognitive_load < 0.3:
            ai_follow_probability -= 0.1

        if scenario_type == "security":
            if "security_coordination" in operator.expertise_areas:
                ai_follow_probability -= 0.15
            elif operator.experience_level == "novice":
                ai_follow_probability += 0.15

        if ai_recommendation.confidence > 0.8:
            ai_follow_probability += 0.1
        elif ai_recommendation.uncertainty > 0.3:
            ai_follow_probability -= 0.2

        ai_follow_probability = float(
            np.clip(ai_follow_probability, 0.0, 1.0)
        )
        follows_ai = random.random() < ai_follow_probability

        if follows_ai:
            action = ai_recommendation.primary_action
            agreement_with_ai = 1.0
            confidence = ai_recommendation.confidence
        elif (
            ai_recommendation.alternative_options
            and random.random() < 0.7
        ):
            chosen_option = random.choice(
                ai_recommendation.alternative_options
            )
            action = chosen_option["action"]
            agreement_with_ai = 0.5
            confidence = chosen_option.get("confidence", 0.5)
        else:
            action = "Maintain current operations and monitor closely"
            agreement_with_ai = 0.0
            confidence = (
                0.6 if operator.experience_level == "expert" else 0.4
            )

        if follows_ai and ai_recommendation.confidence > 0.7:
            sentiment = 0.5
        elif not follows_ai and operator.experience_level == "expert":
            sentiment = 0.3
        else:
            sentiment = 0.0

        if (
            scenario_type == "security"
            and "gate_3_vibration" in ai_recommendation.supporting_data
        ):
            if action == "Schedule immediate maintenance for Gate 3 motor":
                outcome = "successful"
                sentiment = 0.8
            elif "Reduce gate speed by 30%" in action:
                outcome = "partially_successful"
                sentiment = 0.4
            else:
                outcome = "unsuccessful"
                sentiment = -0.6
        elif confidence > 0.7:
            outcome = "successful"
        elif confidence > 0.5:
            outcome = "partially_successful"
        else:
            outcome = "unsuccessful"

        return {
            "action": action,
            "agreement_with_ai": agreement_with_ai,
            "confidence": confidence,
            "sentiment": sentiment,
            "outcome": outcome,
            "reasoning": (
                "Simulated decision based on trust, expertise, "
                "cognitive load, and the scenario rule."
            ),
        }

## 6. Running the Gate 3 example

The inputs below specify vibration, temperature, convoy timing, security level, and scenario complexity. The temperature value of 40.25 is consistent with a 15% increase from a 35-degree baseline; the scenario does not use the temperature value to calculate a prediction. The recommendation rules use the stated scenario values directly.

Because the human decision uses random sampling, the action and trust update can differ between runs. The output describes one simulated interaction. It does not verify the scenario's failure probabilities or demonstrate that an operational plan prevents failure.

In [ ]:
collaboration_system = HumanAICollaboration(symbiotic_facility)

print("Running the Gate 3 high-security scenario")
print("The scenario includes a VIP convoy approaching the facility.")

scenario_data = {
    "gate_3_vibration": 0.45,
    "gate_3_temperature": 40.25,
    "vip_convoy_arrival": 6.0,
    "security_level": "high",
    "complexity": 0.8,
}

expert_operator = symbiotic_facility.operators[0]
interaction = collaboration_system.human_ai_interaction(
    operator=expert_operator,
    scenario_data=scenario_data,
    scenario_type="security",
)

print("\nInteraction results:")
print(
    f"Operator: {interaction['operator_id']} "
    f"({expert_operator.experience_level})"
)
print(
    f"Scenario: {interaction['scenario_type']} - "
    f"cognitive load: {interaction['cognitive_load']:.2f}"
)
print(f"Protocol: {interaction['protocol']}")

ai_recommendation = interaction["ai_recommendation"]
print("\nAI recommendation:")
print(f"Action: {ai_recommendation.primary_action}")
print(f"Recommendation confidence: {ai_recommendation.confidence:.1%}")
print(f"Uncertainty score: {ai_recommendation.uncertainty:.1%}")
print("Reasoning:")
for number, reason in enumerate(ai_recommendation.reasoning, start=1):
    print(f"  {number}. {reason}")

human_decision = interaction["human_decision"]
print("\nSimulated human decision:")
print(f"Action: {human_decision['action']}")
print(f"Agreement with AI: {human_decision['agreement_with_ai']:.1%}")
print(f"Decision confidence: {human_decision['confidence']:.1%}")
print(f"Illustrative outcome: {human_decision['outcome']}")

print("\nTrust update:")
print(f"Trust before: {interaction['trust_before']:.1%}")
print(f"Trust after: {interaction['trust_after']:.1%}")
print(f"Trust change: {interaction['trust_change']:+.1%}")

print(
    "\nInterpretation: the displayed action is a stochastic result of "
    "the simulation rules, not a validated operational recommendation."
)

## 7. Simulating several scenario types

This experiment applies the same interaction procedure to routine monitoring, maintenance planning, emergency response, system optimization, and security coordination. Each scenario is paired with an operator from the facility. The returned records include the scenario type explicitly so that later summaries do not have to infer it from a display name.

The outcome score used in the analysis is 1.0 for a simulated successful outcome, 0.5 for a partially successful outcome, and 0.0 for an unsuccessful outcome. It is an outcome score, not a statistical success rate or an empirical measure of decision quality.

In [ ]:
def run_comprehensive_simulation() -> List[Dict[str, Any]]:
    """Run one illustrative interaction for each listed scenario."""

    print("\nComprehensive symbiotic simulation")

    test_scenarios = [
        {
            "name": "Routine Monitoring",
            "type": "routine",
            "data": {"complexity": 0.3, "system_health": 0.95},
            "operator": 2,
        },
        {
            "name": "Maintenance Planning",
            "type": "maintenance",
            "data": {
                "complexity": 0.6,
                "component_wear": 0.7,
                "cost_threshold": 10000,
            },
            "operator": 1,
        },
        {
            "name": "Emergency Response",
            "type": "emergency",
            "data": {
                "complexity": 0.9,
                "safety_critical": True,
                "time_pressure": 0.8,
            },
            "operator": 0,
        },
        {
            "name": "System Optimization",
            "type": "optimization",
            "data": {
                "complexity": 0.5,
                "demand_forecast": 1.2,
                "efficiency_target": 0.85,
            },
            "operator": 2,
        },
        {
            "name": "Security Coordination",
            "type": "security",
            "data": {
                "complexity": 0.7,
                "threat_level": "medium",
                "coordination_required": True,
            },
            "operator": 0,
        },
    ]

    results = []
    for scenario in test_scenarios:
        print(f"\n--- {scenario['name']} ---")
        operator = symbiotic_facility.operators[scenario["operator"]]
        interaction = collaboration_system.human_ai_interaction(
            operator=operator,
            scenario_data=scenario["data"],
            scenario_type=scenario["type"],
        )

        outcome = interaction["human_decision"]["outcome"]
        outcome_score = {
            "successful": 1.0,
            "partially_successful": 0.5,
            "unsuccessful": 0.0,
        }[outcome]

        result = {
            "scenario": scenario["name"],
            "scenario_type": scenario["type"],
            "operator": operator.operator_id,
            "experience": operator.experience_level,
            "cognitive_load": interaction["cognitive_load"],
            "ai_confidence": interaction["ai_recommendation"].confidence,
            "agreement_with_ai": interaction["human_decision"][
                "agreement_with_ai"
            ],
            "outcome": outcome,
            "outcome_score": outcome_score,
            "trust_before": interaction["trust_before"],
            "trust_after": interaction["trust_after"],
            "trust_change": interaction["trust_change"],
        }
        results.append(result)

        print(f"Operator: {operator.operator_id} ({operator.experience_level})")
        print(f"Cognitive load: {interaction['cognitive_load']:.2f}")
        print(
            "AI agreement: "
            f"{interaction['human_decision']['agreement_with_ai']:.1%}"
        )
        print(f"Illustrative outcome: {outcome}")
        print(f"Trust change: {interaction['trust_change']:+.1%}")

    return results


simulation_results = run_comprehensive_simulation()

## 8. Summarizing simulated performance

### What the summary measures

The analysis calculates the mean outcome score, mean agreement with AI, mean cognitive load, and mean trust change. It also groups the same records by operator experience and scenario type. Since each listed scenario is simulated once, group summaries describe these particular stochastic runs; they do not support general claims about populations or system effectiveness.

The function accepts `results` as an argument and derives its groups from those records. This keeps the summary tied to the data being analyzed.

In [ ]:
def analyze_symbiotic_performance(
    results: List[Dict[str, Any]],
) -> Dict[str, Any]:
    """Summarize outcome scores, agreement, load, and trust changes."""

    if not results:
        raise ValueError("At least one simulation result is required.")

    print("\nSymbiotic partnership simulation analysis")

    outcome_scores = [result["outcome_score"] for result in results]
    average_outcome_score = float(np.mean(outcome_scores))
    average_agreement = float(
        np.mean([result["agreement_with_ai"] for result in results])
    )
    average_cognitive_load = float(
        np.mean([result["cognitive_load"] for result in results])
    )
    average_trust_change = float(
        np.mean([result["trust_change"] for result in results])
    )

    print("\nOverall simulation summaries:")
    print(f"  Number of scenarios: {len(results)}")
    print(f"  Mean outcome score: {average_outcome_score:.2f}")
    print(f"  Mean AI agreement: {average_agreement:.1%}")
    print(f"  Mean cognitive load: {average_cognitive_load:.2f}")
    print(f"  Mean trust change: {average_trust_change:+.1%}")

    experience_groups: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    scenario_groups: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for result in results:
        experience_groups[result["experience"]].append(result)
        scenario_groups[result["scenario_type"]].append(result)

    print("\nSummaries by experience level:")
    for experience, group_results in experience_groups.items():
        group_outcome = np.mean(
            [result["outcome_score"] for result in group_results]
        )
        group_agreement = np.mean(
            [result["agreement_with_ai"] for result in group_results]
        )
        group_trust_change = np.mean(
            [result["trust_change"] for result in group_results]
        )
        print(f"  {experience.title()}:")
        print(f"    Mean outcome score: {group_outcome:.2f}")
        print(f"    Mean AI agreement: {group_agreement:.1%}")
        print(f"    Mean trust change: {group_trust_change:+.1%}")

    print("\nSummaries by scenario type:")
    for scenario_type, group_results in scenario_groups.items():
        group_outcome = np.mean(
            [result["outcome_score"] for result in group_results]
        )
        group_load = np.mean(
            [result["cognitive_load"] for result in group_results]
        )
        print(f"  {scenario_type.title()}:")
        print(f"    Mean outcome score: {group_outcome:.2f}")
        print(f"    Mean cognitive load: {group_load:.2f}")

    trust_improvements = [
        result["trust_change"]
        for result in results
        if result["trust_change"] > 0
    ]
    trust_decreases = [
        result["trust_change"]
        for result in results
        if result["trust_change"] < 0
    ]

    print("\nTrust-change counts:")
    print(f"  Increases: {len(trust_improvements)} scenario records")
    print(f"  Decreases: {len(trust_decreases)} scenario records")
    if trust_improvements:
        print(
            "  Mean increase among increases: "
            f"{np.mean(trust_improvements):+.1%}"
        )
    if trust_decreases:
        print(
            "  Mean change among decreases: "
            f"{np.mean(trust_decreases):+.1%}"
        )

    return {
        "mean_outcome_score": average_outcome_score,
        "mean_agreement": average_agreement,
        "mean_cognitive_load": average_cognitive_load,
        "mean_trust_change": average_trust_change,
        "experience_performance": dict(experience_groups),
        "scenario_performance": dict(scenario_groups),
    }


performance_analysis = analyze_symbiotic_performance(simulation_results)

## 9. Visualizing the simulation records

The four panels show group outcome scores and agreement, cognitive load against agreement, the expert operator's trust across recorded interactions, and outcome score by scenario type. The points and lines summarize the simulation records, not independently measured operational data. With only a few records, a fitted trend line is descriptive and should not be interpreted as evidence of a general relationship.

In [ ]:
def create_symbiotic_visualizations(
    results: List[Dict[str, Any]],
    performance_analysis: Dict[str, Any],
):
    """Plot descriptive summaries of the supplied simulation records."""

    if not results:
        raise ValueError("At least one simulation result is required.")

    figure, axes = plt.subplots(2, 2, figsize=(16, 12))
    figure.suptitle(
        "Human–AI Partnership Simulation",
        fontsize=16,
        fontweight="bold",
    )

    experience_groups = performance_analysis["experience_performance"]
    experience_levels = list(experience_groups)
    outcome_scores = [
        np.mean(
            [result["outcome_score"] for result in experience_groups[level]]
        )
        for level in experience_levels
    ]
    agreement_rates = [
        np.mean(
            [
                result["agreement_with_ai"]
                for result in experience_groups[level]
            ]
        )
        for level in experience_levels
    ]

    ax1 = axes[0, 0]
    x_positions = np.arange(len(experience_levels))
    bar_width = 0.35
    bars_outcome = ax1.bar(
        x_positions - bar_width / 2,
        outcome_scores,
        bar_width,
        label="Mean outcome score",
        color="#2ecc71",
    )
    bars_agreement = ax1.bar(
        x_positions + bar_width / 2,
        agreement_rates,
        bar_width,
        label="Mean AI agreement",
        color="#3498db",
    )
    ax1.set_title("Simulation summaries by experience level")
    ax1.set_xlabel("Experience level")
    ax1.set_ylabel("Score or rate (0–1)")
    ax1.set_xticks(x_positions)
    ax1.set_xticklabels(experience_levels)
    ax1.set_ylim(0, 1.1)
    ax1.legend()
    ax1.grid(axis="y", alpha=0.3)

    for bars in (bars_outcome, bars_agreement):
        for bar in bars:
            height = bar.get_height()
            ax1.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.02,
                f"{height:.2f}",
                ha="center",
                va="bottom",
            )

    ax2 = axes[0, 1]
    cognitive_loads = [result["cognitive_load"] for result in results]
    agreements = [result["agreement_with_ai"] for result in results]
    ax2.scatter(
        cognitive_loads,
        agreements,
        alpha=0.7,
        s=100,
        label="Scenario records",
    )
    if len(results) >= 2 and len(set(cognitive_loads)) >= 2:
        coefficients = np.polyfit(cognitive_loads, agreements, 1)
        trend = np.poly1d(coefficients)
        x_line = np.linspace(0, 1, 100)
        ax2.plot(x_line, trend(x_line), "r--", alpha=0.8, label="Descriptive trend")
    ax2.set_title("Cognitive load and AI agreement")
    ax2.set_xlabel("Cognitive load (0–1)")
    ax2.set_ylabel("AI agreement (0–1)")
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    ax3 = axes[1, 0]
    expert_records = [
        result for result in results if result["operator"] == "OP001"
    ]
    if expert_records:
        trust_values = [expert_records[0]["trust_before"]]
        trust_values.extend(result["trust_after"] for result in expert_records)
        ax3.plot(
            range(len(trust_values)),
            trust_values,
            "g-o",
            linewidth=2,
            markersize=4,
            label="Recorded trust",
        )
        ax3.set_xlabel("Interaction index")
        ax3.set_ylabel("Trust level (0–1)")
        ax3.set_ylim(0, 1)
        ax3.legend()
    else:
        ax3.text(
            0.5,
            0.5,
            "No records for operator OP001",
            ha="center",
            va="center",
            transform=ax3.transAxes,
        )
    ax3.set_title("Trust records for operator OP001")
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    scenario_groups = performance_analysis["scenario_performance"]
    scenario_types = list(scenario_groups)
    scenario_scores = [
        np.mean(
            [result["outcome_score"] for result in scenario_groups[scenario_type]]
        )
        for scenario_type in scenario_types
    ]
    colors = plt.cm.Set3(np.linspace(0, 1, len(scenario_types)))
    bars = ax4.bar(scenario_types, scenario_scores, color=colors)
    ax4.set_title("Outcome score by scenario type")
    ax4.set_xlabel("Scenario type")
    ax4.set_ylabel("Mean outcome score (0–1)")
    ax4.set_ylim(0, 1.1)
    ax4.tick_params(axis="x", rotation=25)
    ax4.grid(axis="y", alpha=0.3)
    for bar, score in zip(bars, scenario_scores):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{score:.2f}",
            ha="center",
            va="bottom",
        )

    figure.tight_layout()
    plt.show()
    return figure


fig = create_symbiotic_visualizations(
    simulation_results,
    performance_analysis,
)

## 10. Interpreting the evidence and limits

The simulation makes the interaction logic inspectable: the recommendation, explanation fields, decision, outcome rule, and trust adjustment can all be traced through the recorded data. Its outputs depend on fixed example rules and random choices. Consequently, they demonstrate how this particular simulation behaves; they do not estimate real-world reliability, prove that human–AI collaboration is superior to human-only or AI-only operation, or establish that the example's operational plan is safe.

The example's proposed combination of speed reduction, backup-gate activation, and maintenance scheduling is not represented as a single decision in the code. The simulated decision selects at most one primary or alternative action. The convoy arrival is stored in the recommendation's supporting data but does not alter the decision rule. Any conclusion about that combined operational plan requires a model that represents its actions, constraints, and consequences.

The explanation fields also have a limited interpretation. The feature weights and attention values are fixed illustrative values rather than explanations generated from a predictive model. A counterfactual entry describes an alternative and, where supplied, its scenario probability; it does not establish causal effects.

A useful reading of the plots is therefore descriptive: they show the outputs of a small set of simulated interactions under the stated assumptions. They are not evidence of statistical relationships or operational effectiveness.

## 11. Why this partnership tier is considered

The human–AI partnership idea addresses situations in which automated pattern recognition and human contextual judgment may play complementary roles. The facility data and protocols illustrate possible divisions of responsibility: an AI system can provide information or advice, while a human retains authority in selected scenarios.

### Potential benefits

- Human expertise can contribute operational context that is absent from a model's input data.
- Explanations and uncertainty communication can support review of recommendations.
- Explicit decision authority and escalation triggers can make responsibilities visible.
- Feedback can be recorded and used in a separately designed learning process.

### Potential challenges

- A human operator may be unavailable, overloaded, or unfamiliar with the AI system.
- Trust may be poorly calibrated, especially when performance changes across situations.
- Human and AI recommendations may conflict, requiring clear escalation and accountability rules.
- Explanations can be misleading if their values do not faithfully reflect the model that produced a recommendation.
- Feedback records alone do not improve an AI system; a learning procedure and evaluation are required.

### When the approach may be relevant

Human oversight may be important for high-consequence, context-dependent, or regulated decisions. Whether a particular partnership design is appropriate depends on operational evidence, safety analysis, accountability requirements, and evaluation against clearly defined baselines. Those evaluations are outside the scope of this illustrative simulation.

### Key takeaways

1. A human–AI interaction can be represented as explicit data structures and decision rules.
2. Cognitive load, trust, and recommendation-following probabilities are distinct quantities; this notebook connects them only through specified simulation rules.
3. A simulated outcome is not an observed operational outcome, and a plausible explanation field is not automatically a faithful model explanation.
4. Context such as a convoy arrival matters only if the decision procedure actually uses it.
5. Claims of collaborative superiority require human-only and AI-only comparisons using appropriate evidence; this notebook does not provide those baselines.

The Centaur-model framing is a useful way to discuss complementary human and AI roles. The code here is a transparent teaching example of those interaction mechanisms, not a deployment-ready gate-control system.